# MCP version of the applicant application

The mcp servers use Rapid api to get linkedin posts, find the relevant posts, send an email to the user for the selected posts, with a curated cover letter and curated c.v


In [20]:
# imports
import os
import requests

from applicant import Applicant
from agents import Agent, Runner, trace
from agents.mcp import MCPServerSse
from templates import (
    evaluation_instructions,
    listing_instructions,
    notification_instructions,
)
from dotenv import load_dotenv
from pypdf import PdfReader
from pathlib import Path
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [ ]:
# set up profile
name = "Denis Gathondu"
denis = Applicant.get(name)
if not denis.summary:
    BASE_DIR = Path.cwd()
    profile_pdf_path = os.path.abspath(
        os.path.join(BASE_DIR, "sandbox", "linkedin.pdf")
    )
    with open(profile_pdf_path, "rb") as pdf_file:
        reader = PdfReader(pdf_file)
        profile = "\n".join([page.extract_text() for page in reader.pages])
    denis.set_summary(profile)


In [ ]:
rapid_api_key = os.getenv("RAPID_API_KEY")
rapid_api_host = os.getenv("RAPID_API_HOST")

url = f"https://{rapid_api_host}/active-jb-7d"

querystring = {
    "limit": "10",
    "offset": "0",
    "advanced_title_filter": "'AI Engineer' | Software:*",
    "location_filter": '"Nairobi"',
    "description_type": "text",
}

headers = {
    "x-rapidapi-key": rapid_api_key,
    "x-rapidapi-host": rapid_api_host,
    "Content-Type": "application/json",
}

response = requests.get(url, headers=headers, params=querystring)


In [ ]:
params = {"url": "http://127.0.0.1:8000/sse"}


In [ ]:
listing_message = f"""
Given the following profile and linked in results. Find me the relevant job posts for {name}.

profile: {denis}
linkedin_results: {response.json()}
"""

async with MCPServerSse(params, client_session_timeout_seconds=30) as listing_server:
    listing_agent = Agent(
        name="listing_agent",
        instructions=listing_instructions(name),
        model="gpt-4o-mini",
        mcp_servers=[listing_server],
    )
    with trace("Listing Agent"):
        result = await Runner.run(listing_agent, listing_message)
    display(Markdown(result.final_output))


In [ ]:
# Evaluator agent
evaluation_message = f"""
Evaluate unevaluated job posts for {name}.

applicant profile: {denis}
"""

async with MCPServerSse(params, client_session_timeout_seconds=30) as eval_server:
    evaluation_agent = Agent(
        name="evaluation_agent",
        instructions=evaluation_instructions(name),
        model="gpt-4o-mini",
        mcp_servers=[eval_server],
    )
    with trace("Evaluation Agent"):
        result = await Runner.run(evaluation_agent, evaluation_message)
    display(Markdown(result.final_output))

In [23]:
# Notification agent
pdf_params = {"url": "http://127.0.0.1:8001/sse"}

async with (
    MCPServerSse(params, client_session_timeout_seconds=60) as notif_server,
    MCPServerSse(pdf_params, client_session_timeout_seconds=60) as pdf_server,
):
    notification_agent = Agent(
        name="notification_agent",
        instructions=notification_instructions(name),
        model="gpt-4o-mini",
        mcp_servers=[notif_server, pdf_server],
    )
    with trace("Notification Agent"):
        result = await Runner.run(
            notification_agent,
            f"Send job application emails for {name}. Profile: {denis}",
        )
    display(Markdown(result.final_output))

### Summary of Job Application Activities

- **Total Applications Sent:** 1
- **Company:** Canonical
- **Role:** Software Engineer - Cloud Images
- **Email Status:** Email sent successfully

No failures were encountered during the process. If you need any further assistance or additional applications, feel free to ask!

In [24]:
denis.list_notifications()

Notifications(notifications=[Notification(evaluation_id=1, notified=False), Notification(evaluation_id=2078454748, notified=False), Notification(evaluation_id=1, notified=True), Notification(evaluation_id=1, notified=True)])